### Building Travel Time Matrix

Builds $t_{ji}$, the estimated driving time from each candidate standby site $j$ (a Brooklyn FDNY firehouse) to the representative point of each ZIP area $i$, for use in `Modelling.ipynb`.

Following the proposal:
1. **Euclidean distance** from each site to each area's representative point (the baseline proxy).
2. **Road-network distance** along the street network (shortest path), assuming one **uniform speed** on every road.
3. **Checks before solving**: each representative point lies inside its own ZIP boundary, every site and area connects to the same road network, and road distances are consistent with Euclidean distances.

**Inputs:** `data/demand.csv`, `data/FDNY_Firehouse_Listing_20261115.csv`, `data/nyc-streets.geojson`, `data/modzcta.geojson`

**Outputs:**
- `data/candidate_sites.csv`: `site_id, name, address, lat, lon, snap_miles`
- `data/travel_times.csv`: long format, one row per (site, area): `site_id, area_id, euclid_miles, network_miles, minutes`

Uses only `numpy`, `pandas`, `scipy` and `matplotlib` (no geopandas or networkx needed).

In [1]:
import json

import numpy as np
import pandas as pd
from matplotlib.path import Path
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, dijkstra
from scipy.spatial import cKDTree

#### 1. Configuration

In [2]:
DATA_DIR = "data"

DEMAND_FILE    = f"{DATA_DIR}/demand.csv"                          # area_id, lat, lon, calls
FIREHOUSE_FILE = f"{DATA_DIR}/FDNY_Firehouse_Listing_20261115.csv"
STREETS_FILE   = f"{DATA_DIR}/nyc-streets.geojson"
BOUNDARY_FILE  = f"{DATA_DIR}/modzcta.geojson"

SITES_OUT        = f"{DATA_DIR}/candidate_sites.csv"
TRAVEL_TIMES_OUT = f"{DATA_DIR}/travel_times.csv"

BOROUGH = "Brooklyn"

# Street network: Kings (047) + Queens (081) counties, so routes near the Brooklyn/Queens line
# are not forced to detour. Census MTFCC road classes considered drivable:
#   S1100 primary road, S1200 secondary road, S1400 local street, S1630 ramp, S1640 service road
STREET_COUNTIES = {"047", "081"}
DRIVABLE_MTFCC  = {"S1100", "S1200", "S1400", "S1630", "S1640"}

SPEED_MPH = 15.0             # uniform driving speed on every road (proposal assumption)
MAX_SNAP_MILES = 0.25        # warn if a point is farther than this from the nearest road node

#### 2. Planar coordinates

At city scale a local equirectangular projection (miles) is accurate to well under 0.1%, so Euclidean distances can be computed directly on `(x, y)`.

In [3]:
EARTH_RADIUS_MI = 3958.8
LAT0 = np.deg2rad(40.65)  # Brooklyn reference latitude

def to_xy(lon, lat):
    # Convert lon/lat degrees to planar (x, y) in miles.
    lon, lat = np.asarray(lon, dtype=float), np.asarray(lat, dtype=float)
    x = EARTH_RADIUS_MI * np.cos(LAT0) * np.deg2rad(lon)
    y = EARTH_RADIUS_MI * np.deg2rad(lat)
    return np.column_stack([x, y])

#### 3. Demand areas ($I$) and candidate sites ($J$)

In [4]:
demand = pd.read_csv(DEMAND_FILE, dtype={"area_id": "string"})
demand = demand.dropna(subset=["lat", "lon"]).reset_index(drop=True)

firehouses = pd.read_csv(FIREHOUSE_FILE)
firehouses.columns = firehouses.columns.str.strip()
firehouses = firehouses.loc[firehouses["Borough"].eq(BOROUGH)].dropna(subset=["Latitude", "Longitude"])

sites = pd.DataFrame({
    "site_id": [f"FH_{k:03d}" for k in range(1, len(firehouses) + 1)],
    "name":    firehouses["FacilityName"].to_numpy(),
    "address": firehouses["FacilityAddress"].to_numpy(),
    "lat":     firehouses["Latitude"].astype(float).to_numpy(),
    "lon":     firehouses["Longitude"].astype(float).to_numpy(),
})

area_xy = to_xy(demand["lon"], demand["lat"])
site_xy = to_xy(sites["lon"], sites["lat"])
print(f"|I| = {len(demand)} areas, |J| = {len(sites)} candidate sites -> {len(demand) * len(sites):,} pairs")

|I| = 37 areas, |J| = 66 candidate sites -> 2,442 pairs


#### 4. Check: representative points lie inside their ZIP boundaries

In [5]:
with open(BOUNDARY_FILE, encoding="utf-8") as f:
    boundaries = json.load(f)

def polygons_for(feature):
    geom = feature["geometry"]
    return [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]

def point_in_area(lon, lat, polys):
    # Inside an outer ring and not inside any of that polygon's holes.
    for rings in polys:
        if Path(rings[0]).contains_point((lon, lat)) and not any(
            Path(hole).contains_point((lon, lat)) for hole in rings[1:]
        ):
            return True
    return False

boundary_by_id = {
    str(feat["properties"].get("modzcta") or feat["properties"].get("MODZCTA")).strip(): polygons_for(feat)
    for feat in boundaries["features"]
}

missing_boundary = sorted(set(demand["area_id"]) - set(boundary_by_id))
assert not missing_boundary, f"No boundary polygon for: {missing_boundary}"

demand["in_own_zip"] = [
    point_in_area(r.lon, r.lat, boundary_by_id[r.area_id]) for r in demand.itertuples()
]
print(f"{demand['in_own_zip'].sum()} of {len(demand)} representative points are inside their own ZIP boundary")
demand.loc[~demand["in_own_zip"]]

37 of 37 representative points are inside their own ZIP boundary


,area_id,lat,lon,calls,in_own_zip


#### 5. Build the road network graph

Nodes are street vertices (coordinates rounded to 6 decimals ≈ 0.1 m so segments meeting at an intersection share a node). Edges are the straight pieces between consecutive vertices, weighted by length in miles. Roads are treated as two-way (the Census file has no one-way information), consistent with the uniform-network assumption.

In [6]:
with open(STREETS_FILE, encoding="utf-8") as f:
    streets = json.load(f)

node_id = {}
edge_u, edge_v = [], []
endpoints = []   # (street name, node) for the two ends of every line piece, used in step 6

def node(coord):
    key = (round(coord[0], 6), round(coord[1], 6))
    if key not in node_id:
        node_id[key] = len(node_id)
    return node_id[key]

n_features = 0
for feat in streets["features"]:
    props, geom = feat["properties"], feat["geometry"]
    if geom is None or props["COUNTYFP"] not in STREET_COUNTIES or props["MTFCC"] not in DRIVABLE_MTFCC:
        continue
    n_features += 1
    lines = [geom["coordinates"]] if geom["type"] == "LineString" else geom["coordinates"]
    for line in lines:
        ids = [node(c) for c in line]
        edge_u.extend(ids[:-1])
        edge_v.extend(ids[1:])
        if props["FULLNAME"]:
            endpoints += [(props["FULLNAME"], ids[0]), (props["FULLNAME"], ids[-1])]

node_lonlat = np.array(list(node_id.keys()))
node_xy = to_xy(node_lonlat[:, 0], node_lonlat[:, 1])

edges = pd.DataFrame({"u": edge_u, "v": edge_v})
edges = edges.loc[edges["u"] != edges["v"]]
edges[["u", "v"]] = np.sort(edges[["u", "v"]].to_numpy(), axis=1)   # undirected: (min, max)
edges = edges.drop_duplicates()                                      # csr_matrix would SUM duplicates
edges["miles"] = np.linalg.norm(node_xy[edges["u"]] - node_xy[edges["v"]], axis=1)
n_nodes = len(node_id)
print(f"{n_features:,} street features -> {n_nodes:,} nodes, {len(edges):,} edges, "
      f"{edges['miles'].sum():,.0f} road miles")

11,623 street features -> 43,855 nodes, 66,642 edges, 4,097 road miles


#### 6. Repair small gaps in the street file

Where a street crosses a divided road, the Census file often splits it into separate pieces that stop at each side, with nothing joining them. For example, 3rd Ave under the Gowanus Expressway is drawn as two local roadways about 130 ft apart. 34th–62nd St each end at both roadways with no link across, so 3rd Ave could not be crossed for ~30 blocks. Two pieces of the **same named street** whose ends lie within `GAP_TOLERANCE_MILES` are therefore joined. Matching on the name avoids attaching dead ends to unrelated roads such as elevated highways.

In [7]:
GAP_TOLERANCE_MILES = 0.03   # ≈ 160 ft

ends = pd.DataFrame(endpoints, columns=["street", "node"]).drop_duplicates()
gap_u, gap_v = [], []
for street, grp in ends.groupby("street"):
    nodes = grp["node"].to_numpy()
    if len(nodes) < 2:
        continue
    for a, b in cKDTree(node_xy[nodes]).query_pairs(GAP_TOLERANCE_MILES):
        gap_u.append(nodes[a])
        gap_v.append(nodes[b])

gaps = pd.DataFrame({"u": gap_u, "v": gap_v}, dtype=int)
gaps[["u", "v"]] = np.sort(gaps[["u", "v"]].to_numpy(), axis=1)
gaps = gaps.drop_duplicates().merge(edges[["u", "v"]], on=["u", "v"], how="left", indicator=True)
gaps = gaps.loc[gaps["_merge"].eq("left_only"), ["u", "v"]]          # skip pairs already linked
gaps["miles"] = np.linalg.norm(node_xy[gaps["u"]] - node_xy[gaps["v"]], axis=1)
edges = pd.concat([edges, gaps], ignore_index=True)

w, u, v = edges["miles"].to_numpy(), edges["u"].to_numpy(), edges["v"].to_numpy()
graph = csr_matrix((np.r_[w, w], (np.r_[u, v], np.r_[v, u])), shape=(n_nodes, n_nodes))
print(f"Added {len(gaps):,} same-street gap links "
      f"(median {gaps['miles'].median() * 5280:.0f} ft, max {gaps['miles'].max() * 5280:.0f} ft)")

Added 329 same-street gap links (median 119 ft, max 158 ft)


#### 7. Check: connectivity, and snap points to the road network

Only the largest connected component is used, so every site can reach every area. Each site and area point is snapped to its nearest road node in that component; the straight-line snap distance is added to the route.

In [8]:
n_comp, labels = connected_components(graph, directed=False)
comp_sizes = np.bincount(labels)
main = np.argmax(comp_sizes)
main_nodes = np.flatnonzero(labels == main)
print(f"{n_comp:,} components; largest has {comp_sizes[main]:,} of {n_nodes:,} nodes "
      f"({comp_sizes[main] / n_nodes:.1%})")

tree = cKDTree(node_xy[main_nodes])

def snap(xy):
    dist, idx = tree.query(xy)
    return main_nodes[idx], dist

site_node, sites["snap_miles"] = snap(site_xy)
area_node, demand["snap_miles"] = snap(area_xy)

for label, df, key in [("site", sites, "site_id"), ("area", demand, "area_id")]:
    far = df.loc[df["snap_miles"] > MAX_SNAP_MILES, key].tolist()
    print(f"Max {label} snap distance: {df['snap_miles'].max():.3f} mi"
          + (f"  -- WARNING, far from road: {far}" if far else ""))

9 components; largest has 43,823 of 43,855 nodes (99.9%)
Max site snap distance: 0.075 mi
Max area snap distance: 0.081 mi


#### 8. Distances and travel times

In [9]:
# Euclidean (straight-line) distance, site j -> area i
euclid = np.linalg.norm(site_xy[:, None, :] - area_xy[None, :, :], axis=2)

# Shortest road path between snapped nodes, plus the snap legs at both ends
road = dijkstra(graph, directed=False, indices=site_node)[:, area_node]
network = road + sites["snap_miles"].to_numpy()[:, None] + demand["snap_miles"].to_numpy()[None, :]
assert np.isfinite(network).all(), "Some (site, area) pairs are not connected"

minutes = 60 * network / SPEED_MPH

travel_times = pd.DataFrame({
    "site_id":       np.repeat(sites["site_id"].to_numpy(), len(demand)),
    "area_id":       np.tile(demand["area_id"].to_numpy(), len(sites)),
    "euclid_miles":  euclid.ravel(),
    "network_miles": network.ravel(),
    "minutes":       minutes.ravel(),
})
travel_times.describe()

,euclid_miles,network_miles,minutes
count,2442.000000,2442.000000,2442.000000
mean,4.165761,4.792297,19.169189
std,2.125108,2.332401,9.329605
min,0.113657,0.232193,0.928771
25%,2.480623,2.968664,11.874658
50%,3.946720,4.578616,18.314465
75%,5.689279,6.446309,25.785238
max,10.830345,12.245615,48.982461


#### 9. Check: road distances vs. Euclidean distances

The ratio network / Euclidean distance (circuity) should be at least 1 and is typically 1.2–1.5 in a city grid. Very large ratios point to a missing road link or a badly snapped point.

In [10]:
far_pairs = travel_times["euclid_miles"] > 0.5        # ratio is noisy for very short pairs
circuity = travel_times.loc[far_pairs, "network_miles"] / travel_times.loc[far_pairs, "euclid_miles"]
print(circuity.describe().round(3))
assert (circuity >= 0.999).all(), "Network distance shorter than straight-line distance"
travel_times.loc[far_pairs].assign(circuity=circuity).nlargest(5, "circuity")

count    2401.000
mean        1.170
std         0.089
min         1.008
25%         1.112
50%         1.154
75%         1.210
max         2.328
dtype: float64


,site_id,area_id,euclid_miles,network_miles,minutes,circuity
2340,FH_064,11211,0.864788,2.013016,8.052063,2.327756
2351,FH_064,11222,2.038720,3.455057,13.820228,1.694719
2182,FH_059,11239,1.123632,1.874205,7.496822,1.667989
2331,FH_064,11201,1.220398,2.033472,8.133888,1.666237
275,FH_008,11217,0.500135,0.827684,3.310734,1.654920


In [11]:
# Sanity check against observed data: closest-firehouse time per area vs. the observed
# EMS travel time (activation -> on scene). These are not directly comparable, since
# ambulances are not always dispatched from the nearest firehouse.
nearest = travel_times.groupby("area_id")["minutes"].min()
observed = pd.read_csv(f"{DATA_DIR}/weekday_daytime_calls.csv", usecols=["incident_travel_tm_seconds_qy"])
print(f"Closest-site time across areas: median {nearest.median():.1f} min, max {nearest.max():.1f} min")
print(f"Observed EMS travel time:        median {observed['incident_travel_tm_seconds_qy'].median() / 60:.1f} min")

Closest-site time across areas: median 2.0 min, max 6.6 min
Observed EMS travel time:        median 8.5 min


#### 10. Save

In [12]:
sites.to_csv(SITES_OUT, index=False)
travel_times.to_csv(TRAVEL_TIMES_OUT, index=False, float_format="%.4f")
print(f"Wrote {SITES_OUT} ({len(sites)} rows) and {TRAVEL_TIMES_OUT} ({len(travel_times):,} rows)")

Wrote data/candidate_sites.csv (66 rows) and data/travel_times.csv (2,442 rows)
